In [1]:
from circular_mode import CircularMode, compute_step_gsm, compute_coupling_matrix
from get_required_modes import analyze_horn_profile_modes
import numpy as np

In [2]:
# define circular junction
#f = 20GHz
freq = 20e9
a = 0.01
b = 0.015

In [3]:
modes_col, mode_ouvertures = analyze_horn_profile_modes(r_profile=np.array([a, b]), freq_hz=freq)

=== ANALYSE MODALE (f0 = 20.0 GHz, Facteur coupure = 4.0) ===
1. COL (r = 10.00 mm) :
   -> 10 modes au total (5 TE + 5 TM)
   -> Propagatifs : 2

2. OUVERTURE (r = 15.00 mm) :
   -> 15 modes au total (8 TE + 7 TM)
   -> Propagatifs : 3


In [4]:
modes_col_ids = [f"{m['type']}{m['m']}{m['n']}" for m in modes_col]
print(modes_col_ids)
#modes_a = [CircularMode(m['type'], m['m'], m['n'], a, freq) for m in modes_col]
modes_a = [CircularMode("TE", m, 1, a, freq) for m in range(10)]

['TE11', 'TE12', 'TE13', 'TE14', 'TE15', 'TM11', 'TM12', 'TM13', 'TM14', 'TM15']


In [5]:
modes_ouverture_ids = [f"{m['type']}{m['m']}{m['n']}" for m in mode_ouvertures]
print(modes_ouverture_ids)
#modes_b = [CircularMode(m['type'], m['m'], m['n'], b, freq) for m in mode_ouvertures]
modes_b = [CircularMode("TE", m, 1, b, freq) for m in range(10)]

['TE11', 'TE12', 'TE13', 'TE14', 'TE15', 'TE16', 'TE17', 'TE18', 'TM11', 'TM12', 'TM13', 'TM14', 'TM15', 'TM16', 'TM17']


In [6]:
M = compute_coupling_matrix(modes_a, modes_b, a)
S, S11, S12, S21, S22 = compute_step_gsm(modes_a, modes_b, M)

In [7]:
w_a_p = np.zeros(S21.shape[1])
w_a_p[0] = 1.
w_b_p = S21 @ w_a_p
print(w_b_p)
print(np.abs(w_b_p))
w_a_m = S11 @ w_a_p
print(w_a_m)
print(np.abs(w_a_m))
w_b_m = np.zeros(S12.shape[1])


[0.63920463+0.01268335j 0.01518506+0.05616921j 0.10390658+0.06227976j
 0.13191973+0.05504463j 0.13072689+0.04538346j 0.11786754+0.03641238j
 0.10149604+0.02887597j 0.08520524+0.02278653j 0.07045047+0.01794931j
 0.05769504+0.01413591j]
[0.63933045 0.05818562 0.12114184 0.14294309 0.13838055 0.12336376
 0.10552378 0.08819953 0.07270108 0.05940152]
[0.25181595+0.08202252j 0.0567069 -0.05949027j 0.05238122-0.07149985j
 0.63320963+0.30041744j 0.04731783+0.00930315j 0.02766985-0.00273687j
 0.0188261 -0.00780007j 0.01368908-0.01031254j 0.01034673-0.01159454j
 0.00802354-0.01220875j]
[0.26483762 0.08218737 0.0886342  0.70086024 0.04822371 0.02780487
 0.02037801 0.01713883 0.01553989 0.01460927]


In [8]:
res_et = M.T @ (w_a_p + w_a_m) - (w_b_p + w_b_m)
print(np.linalg.norm(res_et))

9.74537943758967e-17


In [9]:
Y1 = np.diag([m.Y for m in modes_a])
Y2 = np.diag([m.Y for m in modes_b])
M_T = M.T
res_ht = Y1 @ (w_a_p - w_a_m) - M @ Y2 @ (w_b_p - w_b_m)
print(np.linalg.norm(res_ht))

3.643757743623961e-18


In [10]:
def compute_tangent_field(mode_list: list[CircularMode], w_p, w_m, r, t, z):
    E ,H = np.zeros(3, dtype=np.complex128), np.zeros(3, dtype=np.complex128)
    for i, mode in enumerate(mode_list):
        norm = mode.norm
        e_mode_p, h_mode_p = mode.field_rtz_2(r, t, z)
        E += w_p[i] * e_mode_p * norm
        H += w_p[i] * h_mode_p * norm
        e_mode_m, h_mode_m = mode.field_rtz_2(r, t, z, dir_z=-1.0)
        E += w_m[i] * e_mode_m * norm
        H -= w_m[i] * h_mode_m * norm
    return E, H


In [11]:
z = 0.
r = 0.45 * a
t = np.radians(90.)
Ea, Ha = compute_tangent_field(modes_a, w_a_p, w_a_m, r, t, z)
Eb, Hb = compute_tangent_field(modes_b, w_b_p, w_b_m, r, t, z)
print(Ea)
print(Eb)

[ -24.78135167-18.47132289j -145.26029563 -5.32253943j
    0.         +0.j        ]
[ -0.91956189+2.19021809j -43.12664416-2.31706105j
   0.        +0.j        ]


In [12]:
print(Ha)
print(Hb)

[0.15901885+5.29943605e-03j 0.0076006 -6.76673415e-03j
 0.00017263-3.31363822e-05j]
[ 0.09182323+5.20742271e-03j -0.00203767+5.65860688e-03j
  0.00046881+2.76216945e-07j]
